import


In [1]:
# 📦 Imports
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# 📥 Chargement des données
train_df = pd.read_csv('path')
test_df = pd.read_csv('path')

# 🎯 Cible
y = train_df["Personality"].map({"Introvert": 0, "Extrovert": 1})

# 🔠 Colonnes à encoder
text_cols = ["Stage_fear", "Drained_after_socializing"]
num_cols = ["Going_outside", "Time_spent_Alone", "Post_frequency",
            "Social_event_attendance", "Friends_circle_size"]

# 🔤 Encodage Yes/No
for col in text_cols:
    mapping = {"Yes": 1, "No": 0}
    train_df[col] = train_df[col].map(mapping)
    test_df[col] = test_df[col].map(mapping)

# 🧽 Imputation des valeurs manquantes
imputer = SimpleImputer(strategy="median")
train_df[num_cols] = imputer.fit_transform(train_df[num_cols])
test_df[num_cols] = imputer.transform(test_df[num_cols])

# ✅ Sélection des features
features = text_cols + num_cols
X = train_df[features].fillna(-1)
X_test = test_df[features].fillna(-1)

# 🔀 Séparation entraînement/validation pour early stopping
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

# 🤖 Modèle avec early stopping
model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=1,
    tree_method="hist",
    eval_metric="logloss",
    early_stopping_rounds=20,
    random_state=42
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
best_iter = model.best_iteration + 1
print(f"🌟 Meilleure itération détectée avec early stopping : {best_iter}")

# 📤 Soumission 1 — modèle entraîné avec early stopping
preds_best = model.predict(X_test)
submission_best = pd.DataFrame({
    "id": test_df["id"],
    "Personality": np.where(preds_best == 1, "Extrovert", "Introvert")
})
submission_best.to_csv("submission_best.csv", index=False)
print("✅ submission_best.csv généré")

# 🤖 Modèle réentraîné complet (300 arbres)
final_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=1,
    tree_method="hist",
    eval_metric="logloss",
    random_state=42
)
final_model.fit(X, y)

# 📤 Soumission 2 — modèle complet sans early stopping
preds_full = final_model.predict(X_test)
submission_full = pd.DataFrame({
    "id": test_df["id"],
    "Personality": np.where(preds_full == 1, "Extrovert", "Introvert")
})
submission_full.to_csv("submission_full.csv", index=False)
print("✅ submission_full.csv généré")

# 📊 Soumission 3 — moyenne des probabilités des deux modèles
proba1 = model.predict_proba(X_test)[:, 1]
proba2 = final_model.predict_proba(X_test)[:, 1]
avg_proba = (proba1 + proba2) / 2
avg_preds = np.where(avg_proba >= 0.5, 1, 0)

submission_avg = pd.DataFrame({
    "id": test_df["id"],
    "Personality": np.where(avg_preds == 1, "Extrovert", "Introvert")
})
submission_avg.to_csv("submission_avg.csv", index=False)
print("✅ submission_avg.csv généré")

# 📊 Vérification des distributions
print("\n📈 Distribution des prédictions :")
print("Early stopping :", dict(zip(*np.unique(preds_best, return_counts=True))))
print("Full training  :", dict(zip(*np.unique(preds_full, return_counts=True))))
print("Moyenne proba  :", dict(zip(*np.unique(avg_preds, return_counts=True))))


🌟 Meilleure itération détectée avec early stopping : 135
✅ submission_best.csv généré
✅ submission_full.csv généré
✅ submission_avg.csv généré

📈 Distribution des prédictions :
Early stopping : {np.int64(0): np.int64(1563), np.int64(1): np.int64(4612)}
Full training  : {np.int64(0): np.int64(1563), np.int64(1): np.int64(4612)}
Moyenne proba  : {np.int64(0): np.int64(1563), np.int64(1): np.int64(4612)}
